# Лабораторная работа №2
# Временные ряды

## Цель работы

В данной работе проводится анализ и прогнозирование многомерных временных рядов погодных данных.

Основные задачи:
- получение данных из Excel-файла, включая скрытые листы;
- очистка и унификация временных рядов;
- исследовательский анализ данных;
- feature engineering;
- построение моделей прогнозирования;
- реализация многошагового прогнозирования температуры на 7 дней вперед;
- сравнение recursive и direct forecasting;
- оценка качества моделей.

# 1. Импорт библиотек и настройка окружения

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from openpyxl import load_workbook

from scipy.fft import fft, fftfreq
from scipy.signal import periodogram

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [3]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from catboost import CatBoostRegressor

In [4]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

import optuna

In [5]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))


def directional_accuracy(y_true, y_pred):

    true_direction = np.sign(np.diff(y_true))
    pred_direction = np.sign(np.diff(y_pred))

    return np.mean(true_direction == pred_direction)


def directional_r2(y_true, y_pred):

    true_direction = np.sign(np.diff(y_true))
    pred_direction = np.sign(np.diff(y_pred))

    return r2_score(true_direction, pred_direction)

In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

# 2. Загрузка Excel-файла и первичный анализ данных

In [7]:
excel_path = "../data/have_fun.xlsx"

workbook = load_workbook(excel_path)

sheet_names = workbook.sheetnames

print("Все листы Excel-файла:\n")

for sheet in sheet_names:

    state = workbook[sheet].sheet_state

    print(f"{sheet} | state = {state}")

Все листы Excel-файла:

Ëèñò1 | state = visible
Лист1 | state = visible
ÌÀÈ | state = visible
ýòî | state = visible
ÿ | state = visible
! | state = visible
Ëèñò2_ñòðîêîâûå_NaN_âûáðîñû | state = hidden
Ëèñò3_ñìåøàííûå_òèïû | state = hidden


## 2.1 Загрузка листов в DataFrame

In [8]:
sheets_dict = {}

for sheet in sheet_names:

    df = pd.read_excel(
        excel_path,
        sheet_name=sheet
    )

    sheets_dict[sheet] = df

print(f"Загружено листов: {len(sheets_dict)}")

Загружено листов: 8


## 2.2 Первичный анализ структуры данных

In [9]:
for sheet_name, df in sheets_dict.items():

    print("=" * 70)
    print(f"Лист: {sheet_name}")
    print("=" * 70)

    print(f"Размерность: {df.shape}")

    print("\nПервые строки:")
    display(df.head())

    print("\nТипы данных:")
    print(df.dtypes)

    print("\nКоличество пропусков:")
    print(df.isna().sum())

    print("\n")

Лист: Ëèñò1
Размерность: (125124, 10)

Первые строки:


,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,weathercode,wind_speed_10m,surface_pressure,ds,city
0,3.800000,82,0.0,0.0,0.0,3,19.100000,1010.000000,2020-01-31 02:00:00,Ãåëåíäæèê
1,-3.000000,70,0.0,0.0,0.0,3,24.000000,981.900024,2025-02-28 09:00:00,Áëàãîâåùåíñê
2,-1.300000,86,0.0,0.0,0.0,2,8.800000,1021.200012,2022-01-03 02:00:00,Ãåëåíäæèê
3,19.299999,49,0.0,0.0,0.0,0,22.799999,1016.200012,2020-09-17 06:00:00,Ãåëåíäæèê
4,9.600000,89,0.0,0.0,0.0,0,8.800000,1024.400024,2025-04-16 05:00:00,Ãåëåíäæèê



Типы данных:
temperature_2m                 float64
relative_humidity_2m             int64
precipitation                  float64
rain                           float64
snowfall                       float64
weathercode                      int64
wind_speed_10m                 float64
surface_pressure               float64
ds                      datetime64[us]
city                               str
dtype: object

Количество пропусков:
temperature_2m          1875
relative_humidity_2m       0
precipitation           1247
rain                       0
snowfall                   0
weathercode                0
wind_speed_10m          1252
surface_pressure           0
ds                         0
city                       0
dtype: int64


Лист: Лист1
Размерность: (0, 0)

Первые строки:


""



Типы данных:
Series([], dtype: object)

Количество пропусков:
Series([], dtype: float64)


Лист: ÌÀÈ
Размерность: (0, 0)

Первые строки:


...


Типы данных:
Series([], dtype: object)

Количество пропусков:
Series([], dtype: float64)


Лист: Ëèñò2_ñòðîêîâûå_NaN_âûáðîñû
Размерность: (125169, 25)

Первые строки:


,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,weathercode,wind_speed_10m,surface_pressure,ds,city,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,-6.5,92.0,0,0.0,0.0,3.0,10.8,1000.500000,2019-01-01 00:00:00,Ìîñêâà,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-6.8,92.0,0,0.0,0.0,3.0,10.5,1000.299988,2019-01-01 01:00:00,Ìîñêâà,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-6.8,92.0,0,0.0,0.0,3.0,10.8,1000.000000,2019-01-01 02:00:00,Ìîñêâà,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-5.8,93.0,0,0.0,0.0,3.0,12.2,999.299988,2019-01-01 03:00:00,Ìîñêâà,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,79.276929,93.0,0,0.0,0.0,3.0,14.4,998.599976,2019-01-01 04:00:00,Ìîñêâà,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Типы данных:
temperature_2m                  object
relative_humidity_2m           float64
precipitation                   object
rain                           float64
snowfall                       float64
weathercode                    float64
wind_speed_10m                  object
surface_pressure               float64
ds                      datetime64[us]
city                               str
Unnamed: 10                    float64
Unnamed: 11                        str
Unnamed: 12                        str
Unnamed: 13                        str
Unnamed: 14                     object
Unnamed: 15                     object
Unnamed: 16                     object
Unnamed: 17                     object
Unnamed: 18                     object
Unnamed: 19                     object
Unnamed: 20                     object
Unnamed: 21                     object
Unnamed: 22                     object
Unnamed: 23                     object
Unnamed: 24                        str
dtype: obje

,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,weathercode,wind_speed_10m,surface_pressure,ds,city
0,-1.8,85.0,0,0.0,0.00,3,23.5,1015.200012,2019-01-01 00:00:00,Ñàíêò-Ïåòåðáóðã
1,-1.9,83.0,0,0.0,0.00,3,26,1013.200012,2019-01-01 01:00:00,Ñàíêò-Ïåòåðáóðã
2,-2,85.0,0,0.0,0.00,3,26.200001,1011.700012,2019-01-01 02:00:00,Ñàíêò-Ïåòåðáóðã
3,-2.3,86.0,0,0.0,0.00,3,26.9,1009.900024,2019-01-01 03:00:00,Ñàíêò-Ïåòåðáóðã
4,-2.3,86.0,0.2,0.0,0.14,71,27.9,1007.700012,2019-01-01 04:00:00,Ñàíêò-Ïåòåðáóðã



Типы данных:
temperature_2m                  object
relative_humidity_2m           float64
precipitation                   object
rain                           float64
snowfall                       float64
weathercode                     object
wind_speed_10m                  object
surface_pressure               float64
ds                      datetime64[us]
city                               str
dtype: object

Количество пропусков:
temperature_2m            5
relative_humidity_2m      5
precipitation           249
rain                      0
snowfall                  0
weathercode               0
wind_speed_10m            0
surface_pressure          0
ds                        0
city                      0
dtype: int64




# 3. Очистка и унификация данных
## 3.1 Отбор рабочих листов

In [10]:
valid_sheets = {}

for sheet_name, df in sheets_dict.items():

    if df.shape[0] > 0 and df.shape[1] > 0:
        valid_sheets[sheet_name] = df

print("Рабочие листы:\n")

for sheet in valid_sheets.keys():
    print(sheet)

Рабочие листы:

Ëèñò1
Ëèñò2_ñòðîêîâûå_NaN_âûáðîñû
Ëèñò3_ñìåøàííûå_òèïû


## 3.2 Исправление кодировки строковых данных

In [11]:
def fix_encoding(text):

    if isinstance(text, str):

        try:
            return text.encode("latin1").decode("cp1251")

        except:
            return text

    return text

fixed_sheets = {}

for sheet_name, df in valid_sheets.items():

    fixed_sheet_name = fix_encoding(sheet_name)
    df = df.copy()

    for col in df.columns:
        if df[col].dtype == "object" or df[col].dtype == "str":
            df[col] = df[col].apply(fix_encoding)

    fixed_sheets[fixed_sheet_name] = df

In [12]:
print("Исправленные листы:\n")

for sheet in fixed_sheets.keys():
    print(sheet)

Исправленные листы:

Лист1
Лист2_строковые_NaN_выбросы
Лист3_смешанные_типы


In [13]:
for sheet_name, df in fixed_sheets.items():

    if "city" in df.columns:

        print(f"\n{sheet_name}")

        print(df["city"].unique()[:10])


Лист1
<StringArray>
[    'Геленджик',  'Благовещенск',  'БЛАГОВЕЩЕНСК',   'Благовещенс', 'Благовещенскк',  'благовещенск',     'ГЕЛЕНДЖИК',
     'геленджик',    'Геленджикк',      'Геленджи']
Length: 10, dtype: str

Лист2_строковые_NaN_выбросы
<StringArray>
['Москва', nan, 'Мосва', 'Находка']
Length: 4, dtype: str

Лист3_смешанные_типы
<StringArray>
['Санкт-Петербург', 'Сочи', 'Сычи', 'Счи', 'Соч']
Length: 5, dtype: str


## 3.3 Удаление мусорных столбцов

In [14]:
for sheet_name in fixed_sheets:

    df = fixed_sheets[sheet_name]

    unnamed_cols = [
        col for col in df.columns
        if "Unnamed" in str(col)
    ]

    df = df.drop(columns=unnamed_cols)

    fixed_sheets[sheet_name] = df

    print(f"{sheet_name}: удалено {len(unnamed_cols)} мусорных столбцов")

Лист1: удалено 0 мусорных столбцов
Лист2_строковые_NaN_выбросы: удалено 15 мусорных столбцов
Лист3_смешанные_типы: удалено 0 мусорных столбцов


## 3.4 Приведение типов данных

In [15]:
for sheet_name, df in fixed_sheets.items():

    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)

    object_cols = df.select_dtypes(include=["object"]).columns

    for col in object_cols:

        print(f"\nКолонка: {col}")

        unique_values = df[col].dropna().unique()[:10]

        print(unique_values)


Лист1

Колонка: city
<StringArray>
[    'Геленджик',  'Благовещенск',  'БЛАГОВЕЩЕНСК',   'Благовещенс', 'Благовещенскк',  'благовещенск',     'ГЕЛЕНДЖИК',
     'геленджик',    'Геленджикк',      'Геленджи']
Length: 10, dtype: str

Лист2_строковые_NaN_выбросы

Колонка: temperature_2m
[-6.5 -6.800000190734863 -5.800000190734863 79.27692867104486
 -5.599999904632568 -5.199999809265137 -5.099999904632568
 -5.300000190734863 -5.5 -4.900000095367432]

Колонка: precipitation
[0 0.1000000014901161 0.2000000029802322 0.300000011920929
 0.6000000238418579 0.5 0.4000000059604645 '---' 799.4894255744609
 'нет данных']

Колонка: wind_speed_10m
[10.80000019073486 10.5 12.19999980926514 14.39999961853027 15.5 16
 17.20000076293945 19.39999961853027 22.29999923706055 24]

Колонка: city
<StringArray>
['Москва', 'Мосва', 'Находка']
Length: 3, dtype: str

Лист3_смешанные_типы

Колонка: temperature_2m
[-1.799999952316284 -1.899999976158142 -2 -2.299999952316284
 -2.400000095367432 -2.200000047683716 -2.0

/tmp/ipykernel_72028/1677405558.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=["object"]).columns
/tmp/ipykernel_72028/1677405558.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtype

In [16]:
for sheet_name in fixed_sheets:

    df = fixed_sheets[sheet_name]

    numeric_columns = [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
        "snowfall",
        "weathercode",
        "wind_speed_10m",
        "surface_pressure"
    ]

    for col in numeric_columns:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    fixed_sheets[sheet_name] = df

In [17]:
for sheet_name, df in fixed_sheets.items():

    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)

    print(df.dtypes)


Лист1
temperature_2m                 float64
relative_humidity_2m             int64
precipitation                  float64
rain                           float64
snowfall                       float64
weathercode                      int64
wind_speed_10m                 float64
surface_pressure               float64
ds                      datetime64[us]
city                               str
dtype: object

Лист2_строковые_NaN_выбросы
temperature_2m                 float64
relative_humidity_2m           float64
precipitation                  float64
rain                           float64
snowfall                       float64
weathercode                    float64
wind_speed_10m                 float64
surface_pressure               float64
ds                      datetime64[us]
city                               str
dtype: object

Лист3_смешанные_типы
temperature_2m                 float64
relative_humidity_2m           float64
precipitation                  float64
rain             

## 3.5 Анализ пропусков после очистки

In [18]:
for sheet_name, df in fixed_sheets.items():

    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)

    missing = df.isna().sum()

    print(missing)

    print("\nОбщее количество пропусков:")
    print(missing.sum())


Лист1
temperature_2m          1875
relative_humidity_2m       0
precipitation           1247
rain                       0
snowfall                   0
weathercode                0
wind_speed_10m          1252
surface_pressure           0
ds                         0
city                       0
dtype: int64

Общее количество пропусков:
4374

Лист2_строковые_NaN_выбросы
temperature_2m          3671
relative_humidity_2m    2454
precipitation           3678
rain                    2484
snowfall                2484
weathercode             2485
wind_speed_10m          3094
surface_pressure        2485
ds                      2486
city                    2487
dtype: int64

Общее количество пропусков:
27808

Лист3_смешанные_типы
temperature_2m          1232
relative_humidity_2m       5
precipitation           1232
rain                       0
snowfall                   0
weathercode             1841
wind_speed_10m          1227
surface_pressure           0
ds                         0
city  

## 3.6 Обработка пропусков

In [19]:
missing_before = {}

for sheet_name, df in fixed_sheets.items():
    missing_before[sheet_name] = df.isna().sum().sum()

for sheet_name in fixed_sheets:
    df = fixed_sheets[sheet_name]

    numeric_cols = df.select_dtypes(
        include=["number"]
    ).columns

    df[numeric_cols] = df[numeric_cols].interpolate(
        method="linear"
    )

    fixed_sheets[sheet_name] = df

In [20]:
for sheet_name, df in fixed_sheets.items():

    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)

    current_missing = df.isna().sum().sum()

    print(f"Пропусков было: {missing_before[sheet_name]}")
    print(f"Пропусков стало: {current_missing}")


Лист1
Пропусков было: 4374
Пропусков стало: 0

Лист2_строковые_NaN_выбросы
Пропусков было: 27808
Пропусков стало: 4973

Лист3_смешанные_типы
Пропусков было: 5537
Пропусков стало: 0


## 3.7 Финальная обработка оставшихся пропусков

In [22]:
for sheet_name in fixed_sheets:

    df = fixed_sheets[sheet_name]

    df = df.ffill().bfill()

    fixed_sheets[sheet_name] = df

In [23]:
for sheet_name, df in fixed_sheets.items():

    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)

    print(df.isna().sum().sum())


Лист1
0

Лист2_строковые_NaN_выбросы
0

Лист3_смешанные_типы
0


## 4.1 Сравнение и анализ листов

In [24]:
summary_info = []

for sheet_name, df in fixed_sheets.items():

    info = {
        "sheet": sheet_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    }

    summary_info.append(info)

summary_df = pd.DataFrame(summary_info)

summary_df

,sheet,rows,columns,missing_values,duplicate_rows
0,Лист1,125124,10,0,2383
1,Лист2_строковые_NaN_выбросы,125169,10,0,0
2,Лист3_смешанные_типы,122736,10,0,0


In [25]:
for sheet_name, df in fixed_sheets.items():

    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)

    print(df.describe())


Лист1
       temperature_2m  relative_humidity_2m  precipitation           rain       snowfall    weathercode  \
count   125124.000000         125124.000000  125124.000000  125124.000000  125124.000000  125124.000000   
mean         8.271328             70.459304       0.089957       0.082961       0.004956       9.552700   
min        -41.900002             11.000000       0.000000       0.000000       0.000000       0.000000   
25%          0.800000             58.000000       0.000000       0.000000       0.000000       0.000000   
50%         10.700000             73.000000       0.000000       0.000000       0.000000       2.000000   
75%         19.000000             85.000000       0.000000       0.000000       0.000000       3.000000   
max         37.799999            100.000000      18.000000      18.000000       2.240000      75.000000   
std         14.130598             18.617765       0.430838       0.426297       0.047901      19.795132   

       wind_speed_10m  surfac

## 4.2 Анализ городов и структуры временного ряда

In [31]:
for sheet_name, df in fixed_sheets.items():

    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)

    cities = df["city"].unique()

    print("Города:")

    print(cities)

    print(f"\nКоличество городов: {len(cities)}")


Лист1
Города:
<StringArray>
[    'Геленджик',  'Благовещенск',  'БЛАГОВЕЩЕНСК',   'Благовещенс', 'Благовещенскк',  'благовещенск',     'ГЕЛЕНДЖИК',
     'геленджик',    'Геленджикк',      'Геленджи']
Length: 10, dtype: str

Количество городов: 10

Лист2_строковые_NaN_выбросы
Города:
<StringArray>
['Москва', 'Мосва', 'Находка']
Length: 3, dtype: str

Количество городов: 3

Лист3_смешанные_типы
Города:
<StringArray>
['Санкт-Петербург', 'Сочи', 'Сычи', 'Счи', 'Соч']
Length: 5, dtype: str

Количество городов: 5


In [32]:
for sheet_name, df in fixed_sheets.items():

    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)

    print(df["city"].value_counts())


Лист1
city
Благовещенск     61659
Геленджик        61587
Геленджикк         248
БЛАГОВЕЩЕНСК       245
Геленджи           242
благовещенск       239
Благовещенскк      230
геленджик          228
Благовещенс        226
ГЕЛЕНДЖИК          220
Name: count, dtype: int64

Лист2_строковые_NaN_выбросы
city
Находка    62595
Москва     62573
Мосва          1
Name: count, dtype: int64

Лист3_смешанные_типы
city
Санкт-Петербург    61368
Сочи               61364
Сычи                   2
Счи                    1
Соч                    1
Name: count, dtype: int64


In [33]:
for sheet_name, df in fixed_sheets.items():

    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)

    print("Начало ряда:")
    print(df["ds"].min())

    print("\nКонец ряда:")
    print(df["ds"].max())


Лист1
Начало ряда:
2019-01-01 00:00:00

Конец ряда:
2025-12-31 23:00:00

Лист2_строковые_NaN_выбросы
Начало ряда:
2019-01-01 00:00:00

Конец ряда:
2025-12-31 23:00:00

Лист3_смешанные_типы
Начало ряда:
2019-01-01 00:00:00

Конец ряда:
2025-12-31 23:00:00


## 4.3 Объединение очищенных данных

In [34]:
combined_df = pd.concat(
    fixed_sheets.values(),
    ignore_index=True
)

print(combined_df.shape)

combined_df.head()

(373029, 10)


,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,weathercode,wind_speed_10m,surface_pressure,ds,city
0,3.800000,82.0,0.0,0.0,0.0,3.0,19.100000,1010.000000,2020-01-31 02:00:00,Геленджик
1,-3.000000,70.0,0.0,0.0,0.0,3.0,24.000000,981.900024,2025-02-28 09:00:00,Благовещенск
2,-1.300000,86.0,0.0,0.0,0.0,2.0,8.800000,1021.200012,2022-01-03 02:00:00,Геленджик
3,19.299999,49.0,0.0,0.0,0.0,0.0,22.799999,1016.200012,2020-09-17 06:00:00,Геленджик
4,9.600000,89.0,0.0,0.0,0.0,0.0,8.800000,1024.400024,2025-04-16 05:00:00,Геленджик


## 4.4 Очистка и унификация названий городов

In [35]:
cities = combined_df["city"].unique()

print(cities)

<StringArray>
[      'Геленджик',    'Благовещенск',    'БЛАГОВЕЩЕНСК',     'Благовещенс',   'Благовещенскк',    'благовещенск',
       'ГЕЛЕНДЖИК',       'геленджик',      'Геленджикк',        'Геленджи',          'Москва',           'Мосва',
         'Находка', 'Санкт-Петербург',            'Сочи',            'Сычи',             'Счи',             'Соч']
Length: 18, dtype: str


In [36]:
city_mapping = {

    "геленджик": "Геленджик",
    "Геленджикк": "Геленджик",
    "Геленджи": "Геленджик",
    "ГЕЛЕНДЖИК": "Геленджик",

    "благовещенск": "Благовещенск",
    "БЛАГОВЕЩЕНСК": "Благовещенск",
    "Благовещенскк": "Благовещенск",
    "Благовещенс": "Благовещенск",

    "Мосва": "Москва",

    "Сычи": "Сочи",
    "Счи": "Сочи",
    "Соч": "Сочи"
}

combined_df["city"] = combined_df["city"].replace(
    city_mapping
)

print(combined_df["city"].value_counts())

city
Благовещенск       62599
Находка            62595
Москва             62574
Геленджик          62525
Санкт-Петербург    61368
Сочи               61368
Name: count, dtype: int64


## 4.5 Сортировка временного ряда

In [37]:
combined_df = combined_df.sort_values(
    ["city", "ds"]
)

combined_df = combined_df.reset_index(
    drop=True
)

combined_df.head()

,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,weathercode,wind_speed_10m,surface_pressure,ds,city
0,-13.7,56.0,0.0,0.0,0.0,3.0,12.600000,1008.700012,2019-01-01 00:00:00,Благовещенск
1,-13.2,56.0,0.0,0.0,0.0,3.0,16.299999,1008.500000,2019-01-01 01:00:00,Благовещенск
2,-12.5,56.0,0.0,0.0,0.0,3.0,18.000000,1008.299988,2019-01-01 02:00:00,Благовещенск
3,-12.1,56.0,0.0,0.0,0.0,3.0,18.100000,1007.900024,2019-01-01 03:00:00,Благовещенск
4,-11.9,56.0,0.0,0.0,0.0,3.0,17.600000,1007.500000,2019-01-01 04:00:00,Благовещенск
